# Flipkart Traffic Demand — Per-RoadType LightGBM
**Model:** Separate LightGBM trained on 100% of data (Day 48 + Day 49), one per road type.
**Pipeline:** Data cleaning → Feature engineering → Full-data training → Test submission.

## 1. Imports & Config

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import r2_score
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

SEED = 42
np.random.seed(SEED)

def competition_score(actual, predicted):
    return max(0, 100 * r2_score(actual, predicted))

## 2. Load Data

In [ ]:
df_raw = pd.read_csv("raw.csv")
df = df_raw.copy()
print(f"Shape: {df.shape}")
print(df.head())

## 3. Data Cleaning
Impute `RoadType` from geohash mode, `Weather` as Unknown, `Temperature` from (geohash, hour) medians.

In [ ]:
# ── RoadType: impute from most common value per geohash ──────────────────────
geo_roadtype_mode = (
    df.groupby("geohash")["RoadType"]
    .agg(lambda x: x.mode()[0] if x.notna().any() else np.nan)
)
df["RoadType"] = df["RoadType"].fillna(df["geohash"].map(geo_roadtype_mode))
df["RoadType"] = df["RoadType"].fillna(df["RoadType"].mode()[0])

# ── Weather: small missingness — treat as its own category ───────────────────
df["Weather"] = df["Weather"].fillna("Unknown")

# ── Temperature: median per (geohash, hour), fallback to global median ────────
df["_hour_tmp"] = df["timestamp"].str.split(":").str[0].astype(int)
geo_hour_temp_median = df.groupby(["geohash", "_hour_tmp"])["Temperature"].median()

def fill_temperature(row):
    if pd.isnull(row["Temperature"]):
        return geo_hour_temp_median.get((row["geohash"], row["_hour_tmp"]), np.nan)
    return row["Temperature"]

df["Temperature"] = df.apply(fill_temperature, axis=1)
df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median())
df.drop(columns=["_hour_tmp"], inplace=True)

assert df.isnull().sum().sum() == 0, "Nulls remain after cleaning!"
print("✓ No nulls remaining.")

## 4. Feature Engineering
All features are computed on the **full dataset** (Day 48 + Day 49) since we train on 100% of available data.

In [ ]:
# ── Parse timestamps ─────────────────────────────────────────────────────────
df["hour"]         = df["timestamp"].str.split(":").str[0].astype(int)
df["minute"]       = df["timestamp"].str.split(":").str[1].astype(int)
df["time_minutes"] = df["hour"] * 60 + df["minute"]

# ── Cyclical time encoding (so 23:45 is close to 00:00) ──────────────────────
PERIOD = 1440
df["time_sin"] = np.sin(2 * np.pi * df["time_minutes"] / PERIOD)
df["time_cos"] = np.cos(2 * np.pi * df["time_minutes"] / PERIOD)

# ── Sort chronologically before computing any lags ───────────────────────────
df = df.sort_values(["geohash", "day", "time_minutes"]).reset_index(drop=True)

# ── Geohash 5-char prefix for spatial neighbour feature ──────────────────────
df["geo5"] = df["geohash"].str[:5]

## 5. Full-Dataset Prep for Final Model
Train on all data (Day 48 + 49). Compute target encodings, spatial features, and Yeo-Johnson transform from the complete dataset.

In [ ]:
df_full = df.copy()

# ── Geohash target encoding (smoothed mean demand per geohash) ────────────────
global_demand_mean = df_full["demand"].mean()
stats = df_full.groupby("geohash")["demand"].agg(["mean", "count"])
stats["encoded"] = (
    (stats["count"] * stats["mean"] + 15 * global_demand_mean)
    / (stats["count"] + 15)
)
geo_mean_map = stats["encoded"].to_dict()
df_full["geohash_encoded"] = df_full["geohash"].map(geo_mean_map).fillna(global_demand_mean)

# ── Spatial neighbour: mean demand per (geo5 prefix, hour) ───────────────────
geo5_hour_mean = (
    df_full.groupby(["geo5", "hour"])["demand"]
    .mean()
    .rename("geo5_hour_demand_mean")
    .reset_index()
)
df_full = df_full.merge(geo5_hour_mean, on=["geo5", "hour"], how="left")
df_full["geo5_hour_demand_mean"] = df_full["geo5_hour_demand_mean"].fillna(global_demand_mean)

# ── Binary categorical encodings ─────────────────────────────────────────────
df_full["LargeVehicles_enc"] = (df_full["LargeVehicles"] == "Allowed").astype(int)
df_full["Landmarks_enc"]     = (df_full["Landmarks"] == "Yes").astype(int)

# ── One-hot encode RoadType and Weather ──────────────────────────────────────
df_full = pd.get_dummies(df_full, columns=["RoadType"], drop_first=True)
df_full = pd.get_dummies(df_full, columns=["Weather"],  drop_first=True)

# ── Yeo-Johnson target transform ─────────────────────────────────────────────
pt_yeo = PowerTransformer(method="yeo-johnson", standardize=False)
df_full["yeo_demand"] = pt_yeo.fit_transform(df_full[["demand"]]).flatten()
yeo_min = df_full["yeo_demand"].min()
yeo_max = df_full["yeo_demand"].max()

# ── Native LightGBM geohash category ─────────────────────────────────────────
df_full["geohash_cat"] = df_full["geohash"].astype("category")
known_cats = df_full["geohash_cat"].cat.categories

print(f"df_full shape: {df_full.shape}")
print(f"Yeo-Johnson lambda: {pt_yeo.lambdas_[0]:.4f}")
print(f"yeo_demand skewness: {df_full['yeo_demand'].skew():.4f}")

## 6. Feature List

In [ ]:
FEATURES = [
    "time_sin", "time_cos", "time_minutes", "hour",
    "NumberofLanes", "LargeVehicles_enc", "Landmarks_enc",
    *[c for c in df_full.columns if c.startswith("RoadType_")],
    *[c for c in df_full.columns if c.startswith("Weather_")],
    "Temperature",
    "geohash_encoded", "geo5_hour_demand_mean",
]
# LightGBM native: swap geohash_encoded for the raw category column
FEATURES_LGB = [f for f in FEATURES if f != "geohash_encoded"] + ["geohash_cat"]

missing = [f for f in FEATURES_LGB if f not in df_full.columns]
assert len(missing) == 0, f"Missing features: {missing}"
print(f"Features ({len(FEATURES_LGB)}): {FEATURES_LGB}")

## 7. LightGBM Hyperparameters

In [ ]:
lgb_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.05,
    "num_leaves":       127,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "verbose":          -1,
    "seed":             SEED,
}

## 8. Train Per-RoadType Models on Full Data
One LightGBM per road type (Highway / Street / Residential). Training on 100% of data — no validation holdout, so `num_boost_round` is fixed at 200 to avoid overfitting.

In [ ]:
road_types = ["Highway", "Residential", "Street"]
models_per_road = {}

for road_type in road_types:
    print(f"\nTraining: {road_type}")

    res_mask = df_full.get("RoadType_Residential", pd.Series(0, index=df_full.index)).astype(bool)
    str_mask = df_full.get("RoadType_Street",      pd.Series(0, index=df_full.index)).astype(bool)

    if road_type == "Residential":
        mask = res_mask
    elif road_type == "Street":
        mask = str_mask
    else:  # Highway is the dropped reference category
        mask = ~(res_mask | str_mask)

    sub = df_full[mask]
    print(f"  Rows: {len(sub):,}")

    lgb_data = lgb.Dataset(
        sub[FEATURES_LGB], label=sub["yeo_demand"],
        categorical_feature=["geohash_cat"], free_raw_data=False,
    )
    model = lgb.train(
        params          = lgb_params,
        train_set       = lgb_data,
        num_boost_round = 200,
    )
    models_per_road[road_type] = model
    print(f"  ✓ Done")

print("\nAll models trained.")

## 9. Sanity Check — Validation Score on Day 49
Quick check using Day 49 as a holdout (not used for training above). This gives an indicative score only.

In [ ]:
df_val_check = df_full[df_full["day"] == 49].copy()
y_val_raw = df_val_check["demand"].values

y_pred_check = np.zeros(len(df_val_check))

res_mask_v = df_val_check.get("RoadType_Residential", pd.Series(0, index=df_val_check.index)).astype(bool)
str_mask_v = df_val_check.get("RoadType_Street",      pd.Series(0, index=df_val_check.index)).astype(bool)
hwy_mask_v = ~(res_mask_v | str_mask_v)

for road_type, mask in [("Highway", hwy_mask_v), ("Street", str_mask_v), ("Residential", res_mask_v)]:
    if road_type not in models_per_road:
        continue
    idx = df_val_check.index[mask]
    if len(idx) == 0:
        continue
    yeo_preds = models_per_road[road_type].predict(df_val_check.loc[idx, FEATURES_LGB])
    yeo_preds = np.clip(yeo_preds, yeo_min, yeo_max)
    y_pred_check[mask.values] = np.clip(
        pt_yeo.inverse_transform(yeo_preds.reshape(-1, 1)).flatten(), 0, 1
    )

score = competition_score(y_val_raw, y_pred_check)
print(f"Day 49 indicative score: {score:.4f}")

## 10. Test Pipeline Helper Functions

In [ ]:
def align_columns(df_test_encoded, train_columns):
    """Ensure test matrix has exactly the same columns as training, in order."""
    for col in train_columns:
        if col not in df_test_encoded.columns:
            print(f"  [WARNING] '{col}' missing from test — adding as zeros.")
            df_test_encoded[col] = 0
    extra = set(df_test_encoded.columns) - set(train_columns)
    if extra:
        df_test_encoded.drop(columns=list(extra), inplace=True)
    return df_test_encoded[train_columns]

## 11. Full Test Pipeline

In [ ]:
def run_test_pipeline(
    test_path, df_full, geo_mean_map, geo5_hour_mean,
    models_per_road, pt_yeo, FEATURES_LGB, known_cats,
    global_demand_mean, yeo_min, yeo_max
):
    print("Step 1: Loading test data...")
    df_test = pd.read_csv(test_path)
    print(f"  Shape: {df_test.shape}")

    print("\nStep 2: Cleaning...")
    # RoadType — impute from geohash mode seen in training
    res_mask_tr = df_full.get("RoadType_Residential", pd.Series(0, index=df_full.index)).astype(bool)
    str_mask_tr = df_full.get("RoadType_Street",      pd.Series(0, index=df_full.index)).astype(bool)
    temp_rt = pd.Series("Highway", index=df_full.index)
    temp_rt.loc[res_mask_tr] = "Residential"
    temp_rt.loc[str_mask_tr] = "Street"
    geo_rt_mode = temp_rt.groupby(df_full["geohash"]).agg(lambda x: x.mode()[0])
    df_test["RoadType"] = (
        df_test["RoadType"]
        .fillna(df_test["geohash"].map(geo_rt_mode))
        .fillna("Highway")
    )

    # Weather — unknown for any nulls
    df_test["Weather"] = df_test["Weather"].fillna("Unknown")

    # Temperature — (geohash, hour) median from training, global fallback
    df_test["_hour_tmp"] = df_test["timestamp"].str.split(":").str[0].astype(int)
    geo_hr_temp = df_full.groupby(["geohash", "hour"])["Temperature"].median()
    df_test["Temperature"] = df_test.apply(
        lambda r: geo_hr_temp.get((r["geohash"], r["_hour_tmp"]), np.nan)
        if pd.isnull(r.get("Temperature", np.nan)) else r.get("Temperature", np.nan),
        axis=1
    ).fillna(df_full["Temperature"].median())
    df_test.drop(columns=["_hour_tmp"], inplace=True)

    print("\nStep 3: Feature engineering...")
    df_test["hour"]         = df_test["timestamp"].str.split(":").str[0].astype(int)
    df_test["minute"]       = df_test["timestamp"].str.split(":").str[1].astype(int)
    df_test["time_minutes"] = df_test["hour"] * 60 + df_test["minute"]
    PERIOD = 1440
    df_test["time_sin"] = np.sin(2 * np.pi * df_test["time_minutes"] / PERIOD)
    df_test["time_cos"] = np.cos(2 * np.pi * df_test["time_minutes"] / PERIOD)

    df_test["geohash_encoded"] = (
        df_test["geohash"].map(geo_mean_map).fillna(global_demand_mean)
    )
    df_test["geo5"] = df_test["geohash"].str[:5]
    df_test = df_test.merge(geo5_hour_mean, on=["geo5", "hour"], how="left")
    df_test["geo5_hour_demand_mean"] = df_test["geo5_hour_demand_mean"].fillna(global_demand_mean)

    df_test["LargeVehicles_enc"] = (df_test["LargeVehicles"] == "Allowed").astype(int)
    df_test["Landmarks_enc"]     = (df_test["Landmarks"] == "Yes").astype(int)
    df_test = pd.get_dummies(df_test, columns=["RoadType"], drop_first=True)
    df_test = pd.get_dummies(df_test, columns=["Weather"],  drop_first=True)

    # Align columns to training schema (exclude geohash_cat — added next)
    feat_no_cat = [f for f in FEATURES_LGB if f != "geohash_cat"]
    X_test = align_columns(df_test.copy(), feat_no_cat)
    X_test["geohash_cat"] = pd.Categorical(df_test["geohash"], categories=known_cats)

    print("\nStep 4: Predicting per road type...")
    y_pred = np.zeros(len(df_test))

    res_mask = df_test.get("RoadType_Residential", pd.Series(0, index=df_test.index)).astype(bool)
    str_mask = df_test.get("RoadType_Street",      pd.Series(0, index=df_test.index)).astype(bool)
    hwy_mask = ~(res_mask | str_mask)

    for road_type, mask in [("Highway", hwy_mask), ("Street", str_mask), ("Residential", res_mask)]:
        if road_type not in models_per_road:
            continue
        idx = df_test.index[mask]
        if len(idx) == 0:
            continue
        yeo_preds = models_per_road[road_type].predict(X_test.loc[idx])
        yeo_preds = np.clip(yeo_preds, yeo_min, yeo_max)
        y_pred[mask.values] = np.clip(
            pt_yeo.inverse_transform(yeo_preds.reshape(-1, 1)).flatten(), 0, 1
        )
        print(f"  {road_type}: {mask.sum()} rows predicted")

    print("\nStep 5: Building submission...")
    submission = pd.DataFrame({"Index": df_test["Index"], "demand": y_pred})

    assert submission["demand"].isna().sum() == 0, "NaN predictions!"
    assert (submission["demand"] >= 0).all(),      "Negative predictions!"
    assert (submission["demand"] <= 1).all(),      "Predictions above 1!"
    assert len(submission) == len(df_test),        "Row count mismatch!"

    submission.to_csv("submission.csv", index=False)
    print(f"  ✓ submission.csv saved  ({len(submission):,} rows)")
    print(f"\n  demand stats:")
    print(submission["demand"].describe().round(4))
    return submission

## 12. Generate Submission

In [ ]:
submission = run_test_pipeline(
    test_path          = "test.csv",
    df_full            = df_full,
    geo_mean_map       = geo_mean_map,
    geo5_hour_mean     = geo5_hour_mean,
    models_per_road    = models_per_road,
    pt_yeo             = pt_yeo,
    FEATURES_LGB       = FEATURES_LGB,
    known_cats         = known_cats,
    global_demand_mean = global_demand_mean,
    yeo_min            = yeo_min,
    yeo_max            = yeo_max,
)

## 13. Submission Audit

In [ ]:
sub = pd.read_csv("submission.csv")
print(f"  Rows:        {len(sub):,}")
print(f"  Columns:     {sub.columns.tolist()}")
print(f"  NaNs:        {sub.isnull().sum().values}")
print(f"  Min demand:  {sub['demand'].min():.6f}  (must be >= 0)")
print(f"  Max demand:  {sub['demand'].max():.6f}  (must be <= 1)")
print()
print(sub.head(10))